# Expanded Q4 2025 AI/Tech Earnings Eval

This notebook runs or loads cached outputs for the expanded AI/Tech Q4 2025 earnings universe. Agent outputs are saved under `data/agent_outputs/q4_2025_ai_tech/` by agent type, so future runs can reuse them without calling the models again.


In [1]:
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

CURRENT_DIR = Path.cwd().resolve()
REPO_ROOT = CURRENT_DIR if (CURRENT_DIR / "pyproject.toml").exists() else CURRENT_DIR.parent
sys.path.insert(0, str(REPO_ROOT / "src"))
load_dotenv(REPO_ROOT / ".env", override=True)

from openclam.evaluation import q4_earnings_cache as q4
from openclam.agents.news_macro import news_macro_agent

print(f"Repo root: {REPO_ROOT}")
print(f"Cache root: {REPO_ROOT / q4.DEFAULT_CACHE_ROOT}")
print(f"VERTEX_PROJECT loaded: {bool(os.getenv('VERTEX_PROJECT') or os.getenv('GOOGLE_CLOUD_PROJECT'))}")
print(f"OPENAI_API_KEY loaded: {bool(os.getenv('OPENAI_API_KEY'))}")


Repo root: /Users/zhang/Github/OpenClam_Multi-Agent_Investment_Advisory
Cache root: /Users/zhang/Github/OpenClam_Multi-Agent_Investment_Advisory/data/agent_outputs/q4_2025_ai_tech
VERTEX_PROJECT loaded: True
OPENAI_API_KEY loaded: False


## Configure Universe and Cache

Set `RUN_AGENTS=True` when you want to generate missing cached outputs. Keep `FORCE_RERUN=False` to reuse files that already exist.


In [2]:
STARTER_TICKERS = [
    "AAPL", "MSFT", "GOOGL", "AMZN", "META", "TSLA", "NVDA",
    "AMD", "AVGO", "MU", "TSM", "ASML", "AMAT", "LRCX",
    "ORCL", "DELL", "ANET", "VRT", "CRM", "PLTR",
]

CACHE_ROOT = REPO_ROOT / q4.DEFAULT_CACHE_ROOT
RUN_AGENTS = True
FORCE_RERUN = False

PRE_TRADING_DAYS = 7
POST_TRADING_DAYS = 7
LONG_POST_TRADING_DAYS = 30
NEUTRAL_BAND = 0.02

VERTEX_PROJECT = os.getenv("VERTEX_PROJECT") or os.getenv("GOOGLE_CLOUD_PROJECT")
VERTEX_LOCATION = os.getenv("VERTEX_LOCATION", "us-central1")
VERTEX_MODEL = os.getenv("VERTEX_MODEL", "gemini-2.5-flash")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5-nano")
NEWS_MODEL = os.getenv("NEWS_MODEL", "gpt-5.4-nano")
LLM_PROVIDER = "vertex" if VERTEX_PROJECT else "openai"

earnings_df = q4.q4_2025_ai_tech_earnings_df(STARTER_TICKERS)
earnings_df


,ticker,company,earnings_date,bucket
0,AAPL,Apple,2026-01-29,mega_cap_platform
1,MSFT,Microsoft,2026-01-28,mega_cap_platform
2,GOOGL,Alphabet,2026-02-03,mega_cap_platform
3,AMZN,Amazon,2026-02-05,mega_cap_platform
4,META,Meta Platforms,2026-01-28,mega_cap_platform
5,TSLA,Tesla,2026-01-28,mega_cap_platform
6,NVDA,Nvidia,2026-02-25,mega_cap_platform
7,AMD,Advanced Micro Devices,2026-02-03,ai_semis
8,AVGO,Broadcom,2026-03-05,ai_semis
9,MU,Micron Technology,2025-12-17,ai_semis


## Build Price Eval Table


In [3]:
summary_df, paths_df = q4.build_price_tables(
    earnings_df=earnings_df,
    cache_root=CACHE_ROOT,
    pre_trading_days=PRE_TRADING_DAYS,
    post_trading_days=POST_TRADING_DAYS,
    long_post_trading_days=LONG_POST_TRADING_DAYS,
)

news_macro_agent.format_return_columns(summary_df)


,ticker,company,earnings_date,pre_7d_return,post_1d_return,post_3d_return,post_7d_return,post_30d_return,full_window_return,full_30d_window_return,...,spy_post_30d_return,abnormal_30d_vs_spy,qqq_post_7d_return,abnormal_vs_qqq,qqq_post_30d_return,abnormal_30d_vs_qqq,realized_direction_vs_qqq,realized_30d_direction_vs_qqq,long_horizon_trading_days,bucket
0,AAPL,Apple,2026-01-29,4.69%,0.46%,4.34%,6.43%,-3.07%,11.42%,1.48%,...,-4.57%,1.51%,-2.40%,8.83%,-5.67%,2.60%,up,up,30,mega_cap_platform
1,MSFT,Microsoft,2026-01-28,4.73%,-9.99%,-12.10%,-16.71%,-16.37%,-12.77%,-12.41%,...,-4.22%,-12.15%,-3.72%,-12.99%,-5.68%,-10.69%,down,down,30,mega_cap_platform
2,GOOGL,Alphabet,2026-02-03,3.59%,-1.96%,-4.96%,-9.04%,-9.36%,-5.77%,-6.11%,...,-4.08%,-5.29%,-2.58%,-6.46%,-3.51%,-5.86%,down,down,30,mega_cap_platform
3,AMZN,Amazon,2026-02-05,-8.99%,-5.55%,-7.06%,-9.67%,-7.78%,-17.79%,-16.07%,...,-4.03%,-3.75%,0.72%,-10.39%,-2.51%,-5.27%,down,down,30,mega_cap_platform
4,META,Meta Platforms,2026-01-28,7.82%,10.40%,5.63%,-1.09%,-4.57%,6.64%,2.89%,...,-4.22%,-0.35%,-3.72%,2.64%,-5.68%,1.11%,up,up,30,mega_cap_platform
5,TSLA,Tesla,2026-01-28,-1.38%,-3.45%,-2.24%,-4.72%,-8.45%,-6.03%,-9.71%,...,-4.22%,-4.23%,-3.72%,-0.99%,-5.68%,-2.77%,down,down,30,mega_cap_platform
6,NVDA,Nvidia,2026-02-25,6.97%,-5.46%,-6.69%,-9.07%,-5.95%,-2.73%,0.61%,...,-1.64%,-4.31%,-2.75%,-6.33%,-0.93%,-5.02%,down,down,30,mega_cap_platform
7,AMD,Advanced Micro Devices,2026-02-03,-6.77%,-17.31%,-13.91%,-14.94%,-17.62%,-20.69%,-23.19%,...,-4.08%,-13.54%,-2.58%,-12.36%,-3.51%,-14.11%,down,down,30,ai_semis
8,AVGO,Broadcom,2026-03-05,2.24%,-0.69%,2.95%,-2.36%,22.42%,-0.18%,25.16%,...,4.52%,17.91%,-1.40%,-0.96%,6.69%,15.73%,down,up,30,ai_semis
9,MU,Micron Technology,2025-12-17,-8.67%,10.21%,22.65%,30.58%,94.21%,19.26%,77.38%,...,3.88%,90.33%,3.54%,27.04%,4.42%,89.79%,up,up,30,ai_semis


## Run or Load Cached Agent Outputs


In [4]:
if RUN_AGENTS:
    packets_by_ticker, errors_by_ticker = q4.run_q4_2025_universe_agents(
        earnings_df=earnings_df,
        cache_root=CACHE_ROOT,
        force=FORCE_RERUN,
        lookback_days=14,
        max_news=10,
        llm_provider=LLM_PROVIDER,
        vertex_project=VERTEX_PROJECT,
        vertex_location=VERTEX_LOCATION,
        vertex_model=VERTEX_MODEL,
        openai_model=OPENAI_MODEL,
        news_model=NEWS_MODEL,
    )
else:
    packets_by_ticker = q4.load_cached_packets_by_ticker(CACHE_ROOT)
    errors_by_ticker = {}

print("cached/available tickers:", sorted(packets_by_ticker))
errors_by_ticker


/Users/zhang/opt/anaconda3/envs/agent/lib/python3.10/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.18) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
/Users/zhang/opt/anaconda3/envs/agent/lib/python3.10/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.18) which Google will stop supporting in new releases of google.cloud.resourcemanager_v3 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.resourcemanager_v3 past that date.
  warnings.warn(message, FutureWarning)


cached/available tickers: ['AAPL', 'AMAT', 'AMD', 'AMZN', 'ANET', 'ASML', 'AVGO', 'CRM', 'DELL', 'GOOGL', 'LRCX', 'META', 'MSFT', 'MU', 'NVDA', 'ORCL', 'PLTR', 'TSLA', 'TSM', 'VRT']


{'AAPL': {},
 'MSFT': {},
 'GOOGL': {},
 'AMZN': {},
 'META': {},
 'TSLA': {},
 'NVDA': {},
 'AMD': {},
 'AVGO': {},
 'MU': {},
 'TSM': {},
 'ASML': {},
 'AMAT': {},
 'LRCX': {},
 'ORCL': {},
 'DELL': {},
 'ANET': {},
 'VRT': {},
 'CRM': {},
 'PLTR': {}}

## Run CIO Eval From Cached Packets


In [5]:
cio_eval, cio_results, cio_summary = q4.run_cached_cio_eval(
    summary_df,
    packets_by_ticker,
    cache_root=CACHE_ROOT,
    long_post_trading_days=LONG_POST_TRADING_DAYS,
    neutral_band=NEUTRAL_BAND,
    use_llm_debate=True,
    use_llm_decision=True,
    llm_provider=LLM_PROVIDER,
    debate_model=VERTEX_MODEL if VERTEX_PROJECT else OPENAI_MODEL,
    decision_model=VERTEX_MODEL if VERTEX_PROJECT else OPENAI_MODEL,
    vertex_project=VERTEX_PROJECT,
    vertex_location=VERTEX_LOCATION,
)

news_macro_agent.format_return_columns(cio_eval)


,ticker,company,earnings_date,pre_7d_return,post_1d_return,post_3d_return,post_7d_return,post_30d_return,full_window_return,full_30d_window_return,...,cio_long_term_stance,cio_action,cio_confidence,cio_debate_triggered,cio_debate_conflict_level,cio_reason,cio_short_direction_match,cio_short_direction_match_reason,cio_long_direction_match,cio_long_direction_match_reason
0,AAPL,Apple,2026-01-29,4.69%,0.46%,4.34%,6.43%,-3.07%,11.42%,1.48%,...,Neutral,Watch / No Trade,0.75,True,high,"As CIO, I am ratifying the committee's unanimo...",True,matched,False,neutral missed: abnormal return moved outside ...
1,MSFT,Microsoft,2026-01-28,4.73%,-9.99%,-12.10%,-16.71%,-16.37%,-12.77%,-12.41%,...,Bullish,Accumulate on Weakness,0.85,True,high,The committee reached a unanimous post-debate ...,True,matched,False,missed
2,GOOGL,Alphabet,2026-02-03,3.59%,-1.96%,-4.96%,-9.04%,-9.36%,-5.77%,-6.11%,...,Bullish,Wait for pullback,0.75,False,low,I am adopting a final view that aligns with th...,True,matched,False,missed
3,AMZN,Amazon,2026-02-05,-8.99%,-5.55%,-7.06%,-9.67%,-7.78%,-17.79%,-16.07%,...,Bullish,Wait for pullback to initiate position,0.90,True,high,This decision overrides the deterministic guar...,True,matched,False,missed
4,META,Meta Platforms,2026-01-28,7.82%,10.40%,5.63%,-1.09%,-4.57%,6.64%,2.89%,...,Bullish,Accumulate on weakness,0.75,True,high,An override of the initial purely technical or...,False,missed,True,matched
5,TSLA,Tesla,2026-01-28,-1.38%,-3.45%,-2.24%,-4.72%,-8.45%,-6.03%,-9.71%,...,Bearish,Short / Avoid,0.80,True,high,The decision is based on a unanimous post-deba...,True,matched,True,matched
6,NVDA,Nvidia,2026-02-25,6.97%,-5.46%,-6.69%,-9.07%,-5.95%,-2.73%,0.61%,...,Neutral,Long,0.75,True,high,This decision represents a CIO override of the...,False,missed,False,neutral missed: abnormal return moved outside ...
7,AMD,Advanced Micro Devices,2026-02-03,-6.77%,-17.31%,-13.91%,-14.94%,-17.62%,-20.69%,-23.19%,...,Bullish,Wait for Pullback,0.90,True,high,I am confirming the guardrail's stances but up...,True,matched,False,missed
8,AVGO,Broadcom,2026-03-05,2.24%,-0.69%,2.95%,-2.36%,22.42%,-0.18%,25.16%,...,Bullish,Buy,0.85,False,low,The final decision is an override of the guard...,False,missed,True,matched
9,MU,Micron Technology,2025-12-17,-8.67%,10.21%,22.65%,30.58%,94.21%,19.26%,77.38%,...,Bullish,Long,0.90,False,low,The committee reached a high-conviction bullis...,True,matched,True,matched


In [6]:
cio_summary


{'overall': {'cases': 20,
  'cio_ready_cases': 20,
  'debate_trigger_rate': 0.5,
  'cio_short_direction_match_evaluable': 20,
  'cio_short_direction_match_matched': 10,
  'cio_short_direction_match_accuracy': 0.5,
  'cio_long_direction_match_evaluable': 20,
  'cio_long_direction_match_matched': 12,
  'cio_long_direction_match_accuracy': 0.6},
 'by_bucket': {'ai_infrastructure': {'cases': 4,
   'debate_trigger_rate': 0.5,
   'short_accuracy': 0.5,
   'long_accuracy': 0.75,
   'avg_confidence': 0.8375},
  'ai_semis': {'cases': 7,
   'debate_trigger_rate': 0.14285714285714285,
   'short_accuracy': 0.42857142857142855,
   'long_accuracy': 0.7142857142857143,
   'avg_confidence': 0.85},
  'mega_cap_platform': {'cases': 7,
   'debate_trigger_rate': 0.8571428571428571,
   'short_accuracy': 0.7142857142857143,
   'long_accuracy': 0.2857142857142857,
   'avg_confidence': 0.7928571428571428},
  'software_cloud': {'cases': 2,
   'debate_trigger_rate': 0.5,
   'short_accuracy': 0.0,
   'long_accur

## Inspect Saved Outputs


In [7]:
ticker_to_inspect = "NVDA"
cached = q4.load_cached_ticker(ticker_to_inspect, CACHE_ROOT)
print("saved files:")
for key, value in q4.cached_ticker_paths(ticker_to_inspect, CACHE_ROOT).items():
    print(key, value)

packets = cached.get("packets") or []
packet_df = pd.DataFrame(packets)
display_cols = ["agent_name", "short_term_stance", "long_term_stance", "confidence", "stance_rationale"]
packet_df[[col for col in display_cols if col in packet_df.columns]]


saved files:
news_macro /Users/zhang/Github/OpenClam_Multi-Agent_Investment_Advisory/data/agent_outputs/q4_2025_ai_tech/news_macro/NVDA.json
market_technical /Users/zhang/Github/OpenClam_Multi-Agent_Investment_Advisory/data/agent_outputs/q4_2025_ai_tech/market_technical/NVDA.json
fundamental /Users/zhang/Github/OpenClam_Multi-Agent_Investment_Advisory/data/agent_outputs/q4_2025_ai_tech/fundamental/NVDA.json
packets /Users/zhang/Github/OpenClam_Multi-Agent_Investment_Advisory/data/agent_outputs/q4_2025_ai_tech/packets/NVDA.json
cio /Users/zhang/Github/OpenClam_Multi-Agent_Investment_Advisory/data/agent_outputs/q4_2025_ai_tech/cio/NVDA.json
context /Users/zhang/Github/OpenClam_Multi-Agent_Investment_Advisory/data/agent_outputs/q4_2025_ai_tech/contexts/NVDA.json
errors /Users/zhang/Github/OpenClam_Multi-Agent_Investment_Advisory/data/agent_outputs/q4_2025_ai_tech/errors/NVDA.json


,agent_name,short_term_stance,long_term_stance,confidence,stance_rationale
0,news_macro,Bullish,Bearish,0.65,The short-term stance is Bullish due to a dire...
1,market_technical,Neutral,Bullish,0.85,The dominant technical feature is the conflict...
2,fundamental,Neutral,Bullish,0.60,NVIDIA reported an exceptionally strong quarte...


In [8]:
# Debate trace for one ticker
workflow = cio_results.get(ticker_to_inspect) or q4.load_json(CACHE_ROOT / "cio" / f"{ticker_to_inspect}.json")
pd.DataFrame(workflow["debate"].get("debate_responses", []))


,response_type,agreement,disagreement,revised_short_term_stance,revised_long_term_stance,revised_confidence,evidence_needed
0,agent_revision,{'fundamental_agent': 'I agree with the fundam...,{'market_technical_agent': 'I assign lower wei...,Bullish,Bearish,0.7,"[To invalidate the long-term bearish thesis, I..."
1,revision,{'news_macro': 'I agree with the news_macro ag...,{'news_macro': 'I assign lower weight to the n...,Bullish,Bullish,0.9,[Confirmation of a price breakout above the re...
2,agent_view_revision,[I agree with the news_macro agent's primary e...,[I disagree with the news_macro agent's long-t...,Bullish,Bullish,0.8,[Quantitative data on data center market share...
